# Session 7: Aggregate Functions

This interactive notebook demonstrates SQL's built-in statistical and reporting aggregate functions: `COUNT`, `AVG`, `MAX`, `MIN`, and `SUM`, along with `GROUP BY` categorical partitioning and explicit NULL value handling.

---

## Overview of Tasks
- **Task 1:** Create `Orders` table (`order_id`, `user_name`, `total_amount`, `order_date`) with sample rows containing intentional NULL values.
- **Task 2:** Count orders placed by each user using `COUNT(*)` with `GROUP BY`.
- **Task 3:** Calculate average order total (`AVG`) while safely ignoring NULL values.
- **Task 4:** Extract both highest and lowest order values (`MAX` and `MIN`) within a single dashboard row.
- **Task 5:** Calculate total revenue (`SUM`) applying an explicit `WHERE total_amount IS NOT NULL` pre-filter.

### Task 1: Create 'Orders' Table & Seed Records (with NULLs)

In [ ]:
import sqlite3
import os
import pandas as pd

os.makedirs("database", exist_ok=True)
db_path = os.path.join("database", "orders.db")
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS Orders;")
create_orders_sql = """
CREATE TABLE Orders (
    order_id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_name VARCHAR(50) NOT NULL,
    total_amount DECIMAL(10, 2) NULL,
    order_date DATE NOT NULL
);
"""
cursor.execute(create_orders_sql)

sample_orders = [
    ("Rahul", 1500.00, "2026-10-15"),
    ("Priya", 2499.50, "2026-10-18"),
    ("Rahul", 850.00, "2026-10-20"),
    ("Amit", None, "2026-10-22"),      # Sample NULL record
    ("Priya", 3200.00, "2026-10-25")
]
cursor.executemany("INSERT INTO Orders (user_name, total_amount, order_date) VALUES (?, ?, ?);", sample_orders)
conn.commit()

df_raw = pd.read_sql_query("SELECT * FROM Orders;", conn)
display(df_raw)

### Task 2: Count Orders Per User (COUNT(*) + GROUP BY)

In [ ]:
query_t2 = """
SELECT 
    user_name,
    COUNT(*) AS order_count
FROM Orders
GROUP BY user_name
ORDER BY order_count DESC, user_name ASC;
"""
df_t2 = pd.read_sql_query(query_t2, conn)
display(df_t2)

### Task 3: Calculate Average Order Amount (Ignoring NULLs)
Mathematical verification:
$$\text{Average} = \frac{1500.00 + 2499.50 + 850.00 + 3200.00}{4} = \frac{8049.50}{4} = 2012.38$$

In [ ]:
query_t3 = """
SELECT 
    ROUND(AVG(total_amount), 2) AS average_order_amount
FROM Orders
WHERE total_amount IS NOT NULL;
"""
df_t3 = pd.read_sql_query(query_t3, conn)
display(df_t3)

### Task 4: Highest and Lowest Order Amounts (MAX & MIN in Single Row)

In [ ]:
query_t4 = """
SELECT 
    MAX(total_amount) AS highest_order_amount,
    MIN(total_amount) AS lowest_order_amount
FROM Orders;
"""
df_t4 = pd.read_sql_query(query_t4, conn)
display(df_t4)

### Task 5: Total Sales (SUM with Explicit IS NOT NULL Filter)

In [ ]:
query_t5 = """
SELECT 
    SUM(total_amount) AS total_sales
FROM Orders
WHERE total_amount IS NOT NULL;
"""
df_t5 = pd.read_sql_query(query_t5, conn)
display(df_t5)

conn.close()